In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/amazon"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/amazon/Amazon.csv,Amazon.csv,17377579,1790010070000


In [0]:
from pyspark.sql import functions as F

RAW_PATH = "/Volumes/workspace/default/amazon/Amazon.csv"

amazon_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .csv(RAW_PATH)
)

display(amazon_df)

OrderID,OrderDate,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,Quantity,UnitPrice,Discount,Tax,ShippingCost,TotalAmount,PaymentMethod,OrderStatus,City,State,Country,SellerID
ORD0000001,2023-01-31,CUST001504,Vihaan Sharma,P00014,Drone Mini,Books,BrightLux,3,106.59,0.0,0.0,0.09,319.86,Debit Card,Delivered,Washington,DC,India,SELL01967
ORD0000002,2023-12-30,CUST000178,Pooja Kumar,P00040,Microphone,Home & Kitchen,UrbanStyle,1,251.37,0.05,19.1,1.74,259.64,Amazon Pay,Delivered,Fort Worth,TX,United States,SELL01298
ORD0000003,2022-05-10,CUST047516,Sneha Singh,P00044,Power Bank 20000mAh,Clothing,UrbanStyle,3,35.03,0.1,7.57,5.91,108.06,Debit Card,Delivered,Austin,TX,United States,SELL00908
ORD0000004,2023-07-18,CUST030059,Vihaan Reddy,P00041,Webcam Full HD,Home & Kitchen,Zenith,5,33.58,0.15,11.42,5.53,159.66,Cash on Delivery,Delivered,Charlotte,NC,India,SELL01164
ORD0000005,2023-02-04,CUST048677,Aditya Kapoor,P00029,T-Shirt,Clothing,KiddoFun,2,515.64,0.25,38.67,9.23,821.36,Credit Card,Cancelled,San Antonio,TX,Canada,SELL01411
ORD0000006,2022-12-31,CUST042705,Karan Sharma,P00023,Cookware Set,Books,ReadMore,4,449.73,0.0,215.87,2.74,2017.53,UPI,Delivered,Los Angeles,CA,United States,SELL01494
ORD0000007,2024-09-20,CUST037667,Aarav Verma,P00030,Dress Shirt,Clothing,UrbanStyle,2,219.81,0.2,28.14,14.97,394.81,UPI,Delivered,Chicago,IL,Australia,SELL01676
ORD0000008,2022-11-10,CUST031165,Rohit Kumar,P00028,Jeans,Toys & Games,KiddoFun,2,306.51,0.05,29.12,6.24,617.73,Debit Card,Pending,Denver,CO,India,SELL00510
ORD0000009,2024-06-26,CUST026965,Aman Kapoor,P00031,Kids Toy Car,Sports & Outdoors,Apex,4,146.09,0.0,46.75,7.03,638.14,Debit Card,Delivered,Washington,DC,United States,SELL01895
ORD0000010,2020-05-01,CUST029472,Aarav Reddy,P00001,Wireless Earbuds,Clothing,Apex,2,278.21,0.1,60.09,4.88,565.75,Credit Card,Delivered,Houston,TX,United States,SELL01584


In [0]:
print("Number of rows:", amazon_df.count())
print("Number of columns:", len(amazon_df.columns))

print("\nColumn names:")
print(amazon_df.columns)

print("\nSchema:")
amazon_df.printSchema()

Number of rows: 100000
Number of columns: 20

Column names:
['OrderID', 'OrderDate', 'CustomerID', 'CustomerName', 'ProductID', 'ProductName', 'Category', 'Brand', 'Quantity', 'UnitPrice', 'Discount', 'Tax', 'ShippingCost', 'TotalAmount', 'PaymentMethod', 'OrderStatus', 'City', 'State', 'Country', 'SellerID']

Schema:
root
 |-- OrderID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Tax: double (nullable = true)
 |-- ShippingCost: double (nullable = true)
 |-- TotalAmount: double (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- OrderStatus: string (nullable = true)
 |-- City: string 

In [0]:
from pyspark.sql import functions as F

missing_df = amazon_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in amazon_df.columns
])

display(missing_df)

OrderID,OrderDate,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,Quantity,UnitPrice,Discount,Tax,ShippingCost,TotalAmount,PaymentMethod,OrderStatus,City,State,Country,SellerID
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
total_rows = amazon_df.count()
unique_orders = amazon_df.select("OrderID").distinct().count()

print("Total rows:", total_rows)
print("Unique orders:", unique_orders)
print("Duplicate orders:", total_rows - unique_orders)

Total rows: 100000
Unique orders: 100000
Duplicate orders: 0


In [0]:
# Create Bronze Delta table

(
    amazon_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_amazon_sales")
)

print("Bronze table created successfully!")

Bronze table created successfully!


In [0]:
bronze_df = spark.table("bronze_amazon_sales")

print("Bronze rows:", bronze_df.count())
print("Bronze columns:", len(bronze_df.columns))

display(bronze_df.limit(10))

Bronze rows: 100000
Bronze columns: 20


OrderID,OrderDate,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,Quantity,UnitPrice,Discount,Tax,ShippingCost,TotalAmount,PaymentMethod,OrderStatus,City,State,Country,SellerID
ORD0000001,2023-01-31,CUST001504,Vihaan Sharma,P00014,Drone Mini,Books,BrightLux,3,106.59,0.0,0.0,0.09,319.86,Debit Card,Delivered,Washington,DC,India,SELL01967
ORD0000002,2023-12-30,CUST000178,Pooja Kumar,P00040,Microphone,Home & Kitchen,UrbanStyle,1,251.37,0.05,19.1,1.74,259.64,Amazon Pay,Delivered,Fort Worth,TX,United States,SELL01298
ORD0000003,2022-05-10,CUST047516,Sneha Singh,P00044,Power Bank 20000mAh,Clothing,UrbanStyle,3,35.03,0.1,7.57,5.91,108.06,Debit Card,Delivered,Austin,TX,United States,SELL00908
ORD0000004,2023-07-18,CUST030059,Vihaan Reddy,P00041,Webcam Full HD,Home & Kitchen,Zenith,5,33.58,0.15,11.42,5.53,159.66,Cash on Delivery,Delivered,Charlotte,NC,India,SELL01164
ORD0000005,2023-02-04,CUST048677,Aditya Kapoor,P00029,T-Shirt,Clothing,KiddoFun,2,515.64,0.25,38.67,9.23,821.36,Credit Card,Cancelled,San Antonio,TX,Canada,SELL01411
ORD0000006,2022-12-31,CUST042705,Karan Sharma,P00023,Cookware Set,Books,ReadMore,4,449.73,0.0,215.87,2.74,2017.53,UPI,Delivered,Los Angeles,CA,United States,SELL01494
ORD0000007,2024-09-20,CUST037667,Aarav Verma,P00030,Dress Shirt,Clothing,UrbanStyle,2,219.81,0.2,28.14,14.97,394.81,UPI,Delivered,Chicago,IL,Australia,SELL01676
ORD0000008,2022-11-10,CUST031165,Rohit Kumar,P00028,Jeans,Toys & Games,KiddoFun,2,306.51,0.05,29.12,6.24,617.73,Debit Card,Pending,Denver,CO,India,SELL00510
ORD0000009,2024-06-26,CUST026965,Aman Kapoor,P00031,Kids Toy Car,Sports & Outdoors,Apex,4,146.09,0.0,46.75,7.03,638.14,Debit Card,Delivered,Washington,DC,United States,SELL01895
ORD0000010,2020-05-01,CUST029472,Aarav Reddy,P00001,Wireless Earbuds,Clothing,Apex,2,278.21,0.1,60.09,4.88,565.75,Credit Card,Delivered,Houston,TX,United States,SELL01584


In [0]:
from pyspark.sql import functions as F

summary_df = amazon_df.agg(
    F.count("*").alias("Total_Rows"),
    F.countDistinct("OrderID").alias("Unique_Orders"),
    F.countDistinct("CustomerID").alias("Unique_Customers"),
    F.countDistinct("ProductID").alias("Unique_Products"),
    F.countDistinct("Brand").alias("Unique_Brands"),
    F.countDistinct("Category").alias("Unique_Categories"),
    F.countDistinct("SellerID").alias("Unique_Sellers"),
    F.min("OrderDate").alias("First_Order_Date"),
    F.max("OrderDate").alias("Last_Order_Date"),
    F.sum("TotalAmount").alias("Total_Revenue")
)

display(summary_df)

Total_Rows,Unique_Orders,Unique_Customers,Unique_Products,Unique_Brands,Unique_Categories,Unique_Sellers,First_Order_Date,Last_Order_Date,Total_Revenue
100000,100000,43233,50,10,6,1999,2020-01-01,2024-12-29,9.182564792000014E7


In [0]:
display(
    amazon_df
    .groupBy(F.year("OrderDate").alias("Year"))
    .agg(
        F.count("*").alias("Orders"),
        F.countDistinct("CustomerID").alias("Customers"),
        F.sum("TotalAmount").alias("Revenue")
    )
    .orderBy("Year")
)

Year,Orders,Customers,Revenue
2020,20066,16544,1.852986401999996E7
2021,19890,16446,1.824857481000001E7
2022,20078,16517,1.836724841000009E7
2023,20096,16558,1.851391218999998E7
2024,19870,16360,1.8166048490000047E7


In [0]:
display(
    amazon_df
    .groupBy(
        F.year("OrderDate").alias("Year"),
        F.month("OrderDate").alias("Month")
    )
    .agg(
        F.count("*").alias("Orders"),
        F.countDistinct("CustomerID").alias("Customers"),
        F.sum("TotalAmount").alias("Revenue")
    )
    .orderBy("Year", "Month")
)

Year,Month,Orders,Customers,Revenue
2020,1,1730,1697,1637069.4000000001
2020,2,1490,1473,1339388.2099999995
2020,3,1685,1653,1587125.6600000025
2020,4,1692,1666,1588347.2899999986
2020,5,1672,1639,1533834.1700000016
2020,6,1633,1610,1500269.2900000014
2020,7,1687,1652,1552981.9399999988
2020,8,1783,1753,1640874.9300000006
2020,9,1679,1649,1554682.21
2020,10,1647,1621,1515403.6200000003
